In [6]:
import pandas as pd

df_datos = pd.read_excel(
    'vehiculos_data/Ventas de vehículos eléctricos.xlsx',
    dtype=str,
    skiprows=8
)

In [7]:
df_datos

,Año,Mes,Marca,Familia,Modelo,Segmento,Subsegmento,Provincia,Cantón,Precio,Pais,Combustible,Avaluo,Unidades
0,2020,ene,DAYANG,CHOK CROSS,CHOK CROSS AC 5P 4X2 TA EV,SUV,SMALL,GUAYAS,SIMON BOLIVAR,-,CHINA POPULAR,ELECTRICO BEV,9192,1
1,2020,ene,DAYANG,DY-GD,DY-GD02C AC 2P 4X2 TA EV,AUTOMOVIL,HATCHBACK,GUAYAS,GUAYAQUIL,-,CHINA POPULAR,ELECTRICO BEV,3156,1
2,2020,ene,DAYANG,DY-GD,DY-GD04A AC 2P 4X2 TA EV,AUTOMOVIL,HATCHBACK,GUAYAS,GUAYAQUIL,-,CHINA POPULAR,ELECTRICO BEV,3556,1
3,2020,ene,KAIYUN,PICKMAN,PICKMAN CS 4X2 TA EV,CAMIONETA,CS,AZUAY,CUENCA,-,CHINA POPULAR,ELECTRICO BEV,4640,1
4,2020,ene,KAIYUN,PICKMAN,PICKMAN CS 4X2 TA EV,CAMIONETA,CS,AZUAY,CUENCA,-,CHINA POPULAR,ELECTRICO BEV,4640,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8633,2026,abr,SMART,SMART #1,#1 HX PURE AC 5P 4X2 TA EV,SUV,SMALL,PICHINCHA,QUITO,-,CHINA POPULAR,ELECTRICO BEV,30392,1
8634,2026,abr,VENUCIA,SERIE VX6,VX6 AC 5P 4X2 TA EV,SUV,LOW MEDIUM,PICHINCHA,QUITO,-,CHINA POPULAR,ELECTRICO BEV,28900,1
8635,2026,abr,ZEEKR,ZEEKR 7X,7X LONG RANGE AC 5P 4X2 TA EV,SUV,MEDIUM,PICHINCHA,PEDRO VICENTE MALDONADO,-,CHINA POPULAR,ELECTRICO BEV,59900,1
8636,2026,abr,ZEEKR,ZEEKR 7X,7X LONG RANGE AC 5P 4X2 TA EV,SUV,MEDIUM,PICHINCHA,QUITO,-,CHINA POPULAR,ELECTRICO BEV,59900,1


In [ ]:
import pandas as pd
from typing import List, Optional
from sqlalchemy import create_engine, String, Integer, Float, ForeignKey
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship, Session

# ==========================================
# 1. DEFINICIÓN DE MODELOS ORM
# ==========================================
class Base(DeclarativeBase):
    pass

class Marca(Base):
    __tablename__ = "marca"
    
    id_marca: Mapped[int] = mapped_column(primary_key=True)
    nombre: Mapped[str] = mapped_column(String(50), unique=True)
    origen: Mapped[Optional[str]] = mapped_column(String(50))
    
    modelos: Mapped[List["Modelo"]] = relationship(back_populates="marca", cascade="all, delete-orphan")

class Modelo(Base):
    __tablename__ = "modelo"
    
    id_modelo: Mapped[int] = mapped_column(primary_key=True)
    id_marca: Mapped[int] = mapped_column(ForeignKey("marca.id_marca"))
    nombre: Mapped[str] = mapped_column(String(100))
    tipo_carroceria: Mapped[Optional[str]] = mapped_column(String(30))
    
    marca: Mapped["Marca"] = relationship(back_populates="modelos")
    especificaciones: Mapped[List["EspecificacionTecnica"]] = relationship(back_populates="modelo", cascade="all, delete-orphan")

class EspecificacionTecnica(Base):
    __tablename__ = "especificacion_tecnica"
    
    id_especificacion: Mapped[int] = mapped_column(primary_key=True)
    id_modelo: Mapped[int] = mapped_column(ForeignKey("modelo.id_modelo"))
    version: Mapped[Optional[str]] = mapped_column(String(100))
    autonomia_km: Mapped[Optional[float]] = mapped_column(Float)
    bateria_kwh: Mapped[Optional[float]] = mapped_column(Float)
    potencia_hp: Mapped[Optional[int]] = mapped_column(Integer)
    aceleracion_0_100: Mapped[Optional[float]] = mapped_column(Float)
    
    modelo: Mapped["Modelo"] = relationship(back_populates="especificaciones")

# ==========================================
# 2. CONFIGURACIÓN Y LIMPIEZA DE BASE DE DATOS
# ==========================================
engine = create_engine("sqlite:///vehiculos.db", echo=False) 

# Borrar todas las tablas existentes
Base.metadata.drop_all(engine)
print("Base de datos anterior eliminada correctamente.")

# Volver a crear las tablas en blanco
Base.metadata.create_all(engine)
print("Estructura de la base de datos creada en blanco.")

# ==========================================
# 3. EXTRACCIÓN Y TRANSFORMACIÓN
# ==========================================
# Leer archivo
df_datos = pd.read_excel(
    'vehiculos_data/Ventas de vehículos eléctricos.xlsx',
    dtype=str,
    skiprows=8
)

# Limpiar nombres de columnas para evitar errores por espacios (ej: "Segmento ")
df_datos.columns = df_datos.columns.str.strip()

# Columnas de interés a limpiar
columnas_interes = ['Marca', 'Modelo', 'Segmento']

# Procesar las columnas: quitar espacios, poner en MAYÚSCULAS y asignar nulos reales
for col in columnas_interes:
    if col in df_datos.columns:
        # Convertir a mayúsculas y quitar espacios
        df_datos[col] = df_datos[col].astype(str).str.strip().str.upper()
        # Reemplazar cadenas vacías o 'NAN' generados por pandas con un nulo real de Python
        df_datos[col] = df_datos[col].replace(['NAN', 'NONE', ''], None)

# Eliminar filas donde obligatoriamente falte Marca o Modelo (Segmento sí puede ser nulo)
df_datos = df_datos.dropna(subset=['Marca', 'Modelo'], how='any')

# Eliminar duplicados exactos a nivel de Marca y Modelo
df_datos = df_datos.drop_duplicates(subset=['Marca', 'Modelo'])

# Agrupar jerárquicamente
datos_preparados = []
for marca, grupo in df_datos.groupby('Marca'):
    modelos_lista = []
    
    # Como ya eliminamos duplicados, iteramos fila por fila en el grupo de cada marca
    for _, fila in grupo.iterrows():
        modelos_lista.append({
            "nombre": fila['Modelo'],
            "tipo_carroceria": fila.get('Segmento') # Trae el valor o None si no existe
        })
        
    datos_preparados.append({
        "marca": marca,
        "origen": None,
        "modelos": modelos_lista
    })

# ==========================================
# 4. INGESTA DE DATOS NUEVOS
# ==========================================
with Session(engine) as session:
    for data in datos_preparados:
        # Crear Entidad Padre (Marca)
        nueva_marca = Marca(
            nombre=data["marca"],
            origen=data.get("origen")
        )
        
        # Crear Entidades Hijas (Modelos)
        for mod_data in data["modelos"]:
            nuevo_modelo = Modelo(
                nombre=mod_data["nombre"],
                tipo_carroceria=mod_data.get("tipo_carroceria")
            )
            nueva_marca.modelos.append(nuevo_modelo)
            
        # Añadir a la sesión
        session.add(nueva_marca)
        
    # Guardar en base de datos
    session.commit()
    print("Nuevos datos en mayúsculas cargados exitosamente.")

Base de datos anterior eliminada correctamente.
Estructura de la base de datos creada en blanco.
Nuevos datos en mayúsculas cargados exitosamente (incluyendo Segmento).


In [2]:
datos_preparados

[{'marca': 'AUDI',
  'origen': None,
  'modelos': [{'nombre': 'E-TRON 50 QUATT GENBBE AC 5P 4X4 TA EV',
    'tipo_carroceria': 'SUV'},
   {'nombre': 'E-TRON 50 SPORTBACK QUATT GEABBE AC 5P 4X4 TA EV',
    'tipo_carroceria': 'SUV'},
   {'nombre': 'E-TRON 55 QUATT GENBAE AC 5P 4X4 TA EV',
    'tipo_carroceria': 'SUV'},
   {'nombre': 'E-TRON 55 SPORTBACK SLINE QUATT GEACAE AC 5P 4X4 TA EV',
    'tipo_carroceria': 'SUV'},
   {'nombre': 'E-TRON 55 SPORTBACK QUATT GEABAE AC 5P 4X4 TA EV',
    'tipo_carroceria': 'SUV'},
   {'nombre': 'RS E-TRON GT F83RH7 AC 4P 4X4 TA EV',
    'tipo_carroceria': 'AUTOMOVIL'},
   {'nombre': 'Q8 E-TRON 50 GEGBUB AC 5P 4X4 TA EV', 'tipo_carroceria': 'SUV'},
   {'nombre': 'Q8 E-TRON 50 SPORTBACK GETBUB AC 5P 4X4 TA EV',
    'tipo_carroceria': 'SUV'},
   {'nombre': 'Q8 E-TRON 55 SPORTBACK GETBVC AC 5P 4X4 TA EV',
    'tipo_carroceria': 'SUV'},
   {'nombre': 'Q8 E-TRON 55 QUATT GEGBVC AC 5P 4X4 TA EV',
    'tipo_carroceria': 'SUV'},
   {'nombre': 'Q8 E-TRON 50 SPORT